# Running API with Uvicorn and Ngrok

This notebook shows how to:
1. Start the FastAPI server using uvicorn
2. Expose it publicly using ngrok
3. Test the API endpoints

Useful for testing the API or sharing it temporarily.


In [ ]:
import subprocess
import time
import requests
import json
from threading import Thread
import pyngrok
from pyngrok import ngrok

print("Setup complete")


## Step 1: Start Uvicorn Server

Start the FastAPI server in the background.

In [ ]:
# Configuration
HOST = "127.0.0.1"
PORT = 8000
API_URL = f"http://{HOST}:{PORT}"

# Start uvicorn server in background
uvicorn_process = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", HOST, "--port", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait a bit for server to start
time.sleep(3)

# Check if server is running
try:
    response = requests.get(f"{API_URL}/api/v1/health", timeout=2)
    if response.status_code == 200:
        print(f"Server started successfully on {API_URL}")
    else:
        print(f"Server responded with status {response.status_code}")
except Exception as e:
    print(f"Server might not be ready yet: {e}")
    print("Check if uvicorn is installed: pip install uvicorn")


## Step 2: Expose with Ngrok

Create a public tunnel to your local server.


In [ ]:
# Create ngrok tunnel
# Note: You need ngrok installed and authenticated
# Install: pip install pyngrok
# Or download from: https://ngrok.com/download

try:
    # Start ngrok tunnel
    public_url = ngrok.connect(PORT, "http")
    print(f"Ngrok tunnel created!")
    print(f"Public URL: {public_url}")
    print(f"Local URL: {API_URL}")
    print(f"\nAPI endpoints:")
    print(f"  - Health: {public_url}/api/v1/health")
    print(f"  - Predict: {public_url}/api/v1/predict")
    print(f"  - Docs: {public_url}/docs")
except Exception as e:
    print(f"Error creating ngrok tunnel: {e}")
    print("Make sure ngrok is installed: pip install pyngrok")
    print("Or set up ngrok auth token: ngrok config add-authtoken YOUR_TOKEN")
    public_url = None


## Step 3: Test the API

Test the health endpoint and make a prediction request.


In [ ]:
# Use public URL if ngrok is working, otherwise use local
base_url = str(public_url) if public_url else API_URL
api_base = f"{base_url}/api/v1"

print(f"Testing API at: {api_base}")

# Test health endpoint
try:
    response = requests.get(f"{api_base}/health")
    response.raise_for_status()
    print("Health check passed!")
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f"Health check failed: {e}")


In [ ]:
# Test prediction endpoint
# Replace with your actual S3 paths
payload = {
    "s3_input_path": "username/raw_data/trajectory_data.parquet",
    "username": "test_user",
    "output_filename": "result.parquet"
}

print(f"Sending prediction request to {api_base}/predict")
print(f"Payload: {json.dumps(payload, indent=2)}")
print()

try:
    response = requests.post(f"{api_base}/predict", json=payload)
    response.raise_for_status()
    result = response.json()
    print("Prediction successful!")
    print(json.dumps(result, indent=2))
except requests.exceptions.HTTPError as e:
    print(f"Error: {e}")
    if hasattr(response, 'json'):
        try:
            print(response.json())
        except:
            print(response.text)
except Exception as e:
    print(f"Unexpected error: {e}")


## Step 4: Cleanup

Stop the server and close ngrok tunnel when done.


In [ ]:
# Close ngrok tunnel
if public_url:
    try:
        ngrok.disconnect(public_url)
        print("Ngrok tunnel closed")
    except:
        pass

# Stop uvicorn server
if uvicorn_process:
    uvicorn_process.terminate()
    uvicorn_process.wait()
    print("Uvicorn server stopped")


## Notes

**Prerequisites:**
- Install uvicorn: `pip install uvicorn`
- Install pyngrok: `pip install pyngrok`
- Set up ngrok auth token (if needed): `ngrok config add-authtoken YOUR_TOKEN`
- Or download ngrok binary from https://ngrok.com/download

**Usage:**
1. Make sure your `.env` file is configured with S3 credentials
2. Run the cells above to start server and create tunnel
3. Use the public URL to test from anywhere
4. Remember to stop the server and close tunnel when done

**Security:**
- Ngrok URLs are public - anyone with the URL can access your API
- Use this for testing/development only
- For production, use proper hosting (AWS, GCP, etc.)
